# Queries
This notebook contains a set of DMS query examples that can be used as templates.

In [1]:
import json
import os
import time
from collections import defaultdict
from pathlib import Path

from dotenv import load_dotenv

from cognite.client import CogniteClient, global_config
from cognite.client.config import ClientConfig
from cognite.client.credentials import OAuthClientCredentials
from cognite.client.data_classes.data_modeling import (
    EdgeListWithCursor,
    InstanceSort,
    NodeId,
    NodeListWithCursor,
    ViewId,
    DirectRelationReference,
)
from cognite.client.data_classes.data_modeling.query import (
    NodeResultSetExpression,
    Query,
    QueryResult,
    Select,
    SourceSelector,
)
from cognite.client.data_classes.filters import (
    And,
    ContainsAll,
    ContainsAny,
    Equals,
    HasData,
    Nested,
    Prefix,
    SpaceFilter,
)
from cognite.client.exceptions import CogniteAPIError
from tenacity import (
    retry,
    retry_if_exception,
    stop_after_attempt,
    wait_exponential_jitter,
)

## Basic setup

In [2]:
# Locate the .env file by walking up from the current working directory until we find one.
# Notebooks set cwd to wherever Jupyter was launched, so a fixed `parents[N]` is fragile.
def _find_env_file(start: Path | None = None) -> Path | None:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        env_path = candidate / ".env"
        if env_path.is_file():
            return env_path
    return None


env_file = _find_env_file()
if env_file is None:
    raise FileNotFoundError(
        "Could not find a .env file by walking up from cwd. "
        "Place a .env at the repo root or set the CDF_* / IDP_* env vars in your shell."
    )
print(f"Loading credentials from {env_file}")
load_dotenv(env_file, override=False)

project = os.getenv("CDF_PROJECT")
cluster = os.getenv("CDF_CLUSTER")
client_id = os.getenv("CDF_CLIENT_ID") or os.getenv("IDP_CLIENT_ID")
client_secret = os.getenv("CDF_CLIENT_SECRET") or os.getenv("IDP_CLIENT_SECRET")
tenant_id = os.getenv("CDF_TENANT_ID") or os.getenv("IDP_TENANT_ID")
base_url = os.getenv("CDF_BASE_URL") or os.getenv("CDF_URL") or (f"https://{cluster}.cognitedata.com" if cluster else None)

missing = [
    name for name, val in {
        "CDF_PROJECT": project,
        "CDF_CLIENT_ID / IDP_CLIENT_ID": client_id,
        "CDF_CLIENT_SECRET / IDP_CLIENT_SECRET": client_secret,
        "CDF_TENANT_ID / IDP_TENANT_ID": tenant_id,
        "CDF_BASE_URL / CDF_URL / CDF_CLUSTER": base_url,
    }.items()
    if not val
]
if missing:
    raise RuntimeError(
        f"Missing required environment variables: {', '.join(missing)}.\n"
        f"Checked .env: {env_file}\n"
        "Make sure these are defined there or exported in your shell."
    )

credentials = OAuthClientCredentials(
    token_url=f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
    client_id=client_id,
    client_secret=client_secret,
    scopes=[f"{base_url}/.default"],
)

# Increase SDK-level retry budget so transient 429/5xx are handled before they reach user code.
# The SDK already retries 429 and 5xx with backoff; we just give it more headroom.
global_config.max_retries = 10
global_config.max_retry_backoff = 30
global_config.disable_pypi_version_check = True

client = CogniteClient(ClientConfig(
    client_name="cdf-query-examples",
    project=project,
    credentials=credentials,
    base_url=base_url,
))

# Merge (don't overwrite) headers so SDK defaults like x-cdp-sdk / User-Agent are preserved.
# cdf-version: alpha unlocks alpha features on /query (e.g. the "debug" envelope used below).
client.config.headers = {**(client.config.headers or {}), "cdf-version": "alpha"}
print("Client configured with alpha features enabled and SDK retries tuned (max_retries=10).")

Loading credentials from C:\Users\JanIngeBergseth\OneDrive - Cognite AS\Documents\GitHub\library\.env
Client configured with alpha features enabled and SDK retries tuned (max_retries=10).


# Retry helpers (408 / 429 / 5xx)

These wrappers handle the two failure modes you'll hit most often when running DMS queries:

- **408 Request Timeout** — usually caused by oversized payloads (`properties=["*"]`), high `limit`, or expensive joins. The right long-term fix is to narrow the query; the retry only buys time.
- **429 Too Many Requests** — token-bucket throttling. Exponential backoff with jitter clears it.

The decorator below retries on `{408, 425, 429, 500, 502, 503, 504}` with jittered exponential backoff. The Cognite SDK also retries internally (configured above via `global_config`); these wrappers add a second layer for the user-facing call.

In [3]:
RETRYABLE_CODES = {408, 425, 429, 500, 502, 503, 504}


def is_retryable_exception(e: BaseException) -> bool:
    """Retry on transient transport / throttling errors."""
    return isinstance(e, CogniteAPIError) and e.code in RETRYABLE_CODES


retry_cognite = retry(
    reraise=True,
    stop=stop_after_attempt(5),
    retry=retry_if_exception(is_retryable_exception),
    wait=wait_exponential_jitter(initial=1, max=30, jitter=2),
)


@retry_cognite
def run_query(client: CogniteClient, query: Query) -> QueryResult:
    """Run a /query against the data model with retry on 408/429/5xx."""
    return client.data_modeling.instances.query(query=query)


@retry_cognite
def run_sync(client: CogniteClient, query: Query) -> QueryResult:
    """Run a /sync (cursor-based) query with retry on 408/429/5xx."""
    return client.data_modeling.instances.sync(query=query)


def log_api_error(e: CogniteAPIError) -> None:
    """Print enough context to correlate with CDF backend logs."""
    print(f"CogniteAPIError code={e.code} x_request_id={getattr(e, 'x_request_id', None)}: {e.message}")


def _summarize(
    label: str,
    nodes,
    vid: ViewId,
    max_rows: int = 10,
    extra_fields: list[str] | None = None,
) -> None:
    """Compact, capped instance output for notebook demos."""
    from itertools import islice

    print(f"{label} (count={len(nodes)}):")
    for n in islice(nodes, max_rows):
        props = n.properties.get(vid, {}) or {}
        parts = [
            f"{n.external_id}",
            f"name={props.get('name')!r}",
            f"description={props.get('description')!r}",
        ]
        for key in (extra_fields or []):
            parts.append(f"{key}={props.get(key)!r}")
        print("  " + "  ".join(parts))
    if len(nodes) > max_rows:
        print(f"   ... and {len(nodes) - max_rows} more")


def _summarize_raw_rows(
    label: str,
    rows: list[dict],
    view_space: str,
    view_external_id: str,
    view_version: str,
    max_rows: int = 10,
) -> None:
    """Compact, capped output for raw JSON /query rows."""
    from itertools import islice

    print(f"{label} (count={len(rows)}):")
    view_key = f"{view_external_id}/{view_version}"
    for row in islice(rows, max_rows):
        props = ((row.get("properties") or {}).get(view_space, {}) or {}).get(view_key, {}) or {}
        print(
            f"  {row.get('externalId')}  "
            f"name={props.get('name')!r}  "
            f"description={props.get('description')!r}"
        )
    if len(rows) > max_rows:
        print(f"   ... and {len(rows) - max_rows} more")

# Setup

Define the instance space, the model space, the model version, and all `ViewId`s used by the queries below. Centralising these constants makes it trivial to retarget the notebook at a different deployment.

In [4]:
# Target model in this project: dm_dom_oil_and_gas : dm_oil_and_gas_domain_model / v1
# Adjust INST_SP if your instance space differs from the model space.
INST_SP = "inst_location"    # instance space
MODEL_SP_CDM = "cdf_cdm"  # model space (where views/containers live)
MODEL_SP_IDM = "cdf_idm"  # model space (where views/containers live)
MODEL_SP_OG = "dm_dom_oil_and_gas" # model space for Oil & Gas template

MODEL_VERSION = "v1"

# Project views (resolved from `client.data_modeling.views.list(space=MODEL_SP)`).
# `Tag` is this model's "asset" equivalent (functional tag). There is no Activity view -
# closest analogues are WorkOrder / WorkOrderOperation.
asset_vid = ViewId(space=MODEL_SP_CDM, external_id="CogniteAsset", version=MODEL_VERSION)
ts_vid = ViewId(space=MODEL_SP_CDM, external_id="CogniteTimeSeries", version=MODEL_VERSION)
file_vid = ViewId(space=MODEL_SP_CDM, external_id="CogniteFile", version=MODEL_VERSION)
eq_vid = ViewId(space=MODEL_SP_CDM, external_id="CogniteEquipment", version=MODEL_VERSION)
wo_vid = ViewId(space=MODEL_SP_IDM, external_id="CogniteMaintenanceOrder", version=MODEL_VERSION)
wo_op_vid = ViewId(space=MODEL_SP_IDM, external_id="CogniteOperation", version=MODEL_VERSION)
notification_vid = ViewId(space=MODEL_SP_IDM, external_id="CogniteNotification", version=MODEL_VERSION)

tag_vid = ViewId(space=MODEL_SP_OG, external_id="Tag", version=MODEL_VERSION)


Optional sanity check: list all views in the model space.

In [5]:
# Inspect the ViewIds defined in the Setup cell. For each one, retrieve the full
# view definition from CDF and print its properties so you know what's available
# for filter / select / sort. Driven by the variables in Setup so renaming a view
# in one place is enough.
view_ids = {
    "asset_vid": asset_vid,
    "tag_vid": tag_vid,
    "ts_vid": ts_vid,
    "file_vid": file_vid,
    "eq_vid": eq_vid,
    "wo_vid": wo_vid,
    "wo_op_vid": wo_op_vid,
    "notification_vid": notification_vid,
}

retrieved = client.data_modeling.views.retrieve(list(view_ids.values()))
retrieved_by_id = {(v.space, v.external_id, v.version): v for v in retrieved}

# Cache property metadata per view for downstream query-design / debugging.
# Schema: view_properties[var_name] = {prop_name: {"type": ..., "is_list": bool,
#         "is_direct_relation": bool, "is_reverse_direct_relation": bool,
#         "container": (space, external_id) | None}}
view_properties: dict[str, dict[str, dict]] = {}

for var_name, vid in view_ids.items():
    view = retrieved_by_id.get((vid.space, vid.external_id, vid.version))
    if view is None:
        print(f"{var_name} -> {vid.space}:{vid.external_id}/{vid.version}  (NOT FOUND)")
        view_properties[var_name] = {}
        continue

    props: dict[str, dict] = {}
    for prop_name, prop in view.properties.items():
        prop_type = getattr(prop, "type", None)
        cls_name = prop.__class__.__name__
        container = getattr(prop, "container", None)
        props[prop_name] = {
            "type": str(prop_type) if prop_type is not None else cls_name,
            "is_list": getattr(prop, "list", False),
            "is_direct_relation": cls_name == "MappedPropertyApply" and "direct" in str(prop_type).lower()
                or "direct" in cls_name.lower(),
            "is_reverse_direct_relation": "ReverseDirectRelation" in cls_name,
            "container": (container.space, container.external_id) if container is not None else None,
        }
    view_properties[var_name] = props
    print(f"{var_name} -> {vid.space}:{vid.external_id}/{vid.version}  ({len(props)} properties)")

# Quick-access helpers for query design:
#   view_properties["asset_vid"]["parent"]            -> dict of metadata
#   [p for p, m in view_properties["asset_vid"].items() if m["is_direct_relation"]]
#   [p for p, m in view_properties["asset_vid"].items() if m["is_reverse_direct_relation"]]
#   [p for p, m in view_properties["asset_vid"].items() if m["is_list"]]


asset_vid -> cdf_cdm:CogniteAsset/v1  (23 properties)
tag_vid -> dm_dom_oil_and_gas:Tag/v1  (50 properties)
ts_vid -> cdf_cdm:CogniteTimeSeries/v1  (19 properties)
file_vid -> cdf_cdm:CogniteFile/v1  (18 properties)
eq_vid -> cdf_cdm:CogniteEquipment/v1  (18 properties)
wo_vid -> cdf_idm:CogniteMaintenanceOrder/v1  (24 properties)
wo_op_vid -> cdf_idm:CogniteOperation/v1  (27 properties)
notification_vid -> cdf_idm:CogniteNotification/v1  (21 properties)


# Search endpoint

Search is tokenized full-text lookup (not regex; see the docs for the exact tokenization rules).

This endpoint targets the Elasticsearch backend where *every text property is indexed*, so it is typically more performant than `/query` for keyword-style lookups.

Caveat: search can only match `text`-typed properties.

## Search-first workflows (recommended production pattern)

For efficient retrieval from DMS, start with:

1. `instances.search` to find/rank candidate nodes quickly.
2. Optional `filter=` to narrow scope (space, sourceContext, subtree, etc.).
3. Use resulting node references (`space`, `externalId`) as anchors for the next query/traversal.

This usually performs better and is easier to reason about than broad `list(...)` scans.

### List all assets safely (cursor pagination, no functional filter)

For large volumes, avoid `limit=-1` one-shot reads. Use cursor pagination with a fixed page size (e.g. 1000) and retry on transient API errors. This pattern is resilient to 429 throttling and 5xx/timeout issues.

Notes:
- `run_sync(...)` in this notebook is retry-wrapped (`retry_cognite`), so transient 429/5xx are retried automatically.
- This example uses **no business filter** (only scope guards: `space` + `HasData(asset_vid)`), and drains pages until cursors are exhausted.

In [6]:
# Cursor-paginated retrieval of all assets in INST_SP (no business filter).
# Page size 1000 is a practical default for throughput vs timeout risk.

PAGE_SIZE = 1000

all_assets_query = Query(
    with_={
        "assets": NodeResultSetExpression(
            filter=And(
                Equals(["node", "space"], value={"parameter": "space"}),
                HasData(views=[asset_vid]),
            ),
            limit=PAGE_SIZE,
            # Note: /sync does not support `sort` in node table expressions.
            # If you need deterministic display order, sort the collected list after retrieval.
        ),
    },
    select={
        "assets": Select([
            SourceSelector(asset_vid, ["name", "description", "tags"]),
        ]),
    },
    parameters={"space": INST_SP},
)

all_assets = []
page = 0

while True:
    # run_sync is retry-wrapped in this notebook (429/5xx/timeout transient handling)
    res = run_sync(client, all_assets_query)

    batch = res.data.get("assets", [])
    if not batch:
        break

    page += 1
    all_assets.extend(batch)
    print(f"Page {page}: fetched {len(batch)} rows (running total={len(all_assets)})")

    all_assets_query.cursors = res.cursors

# Compact preview (first 10) using shared helper
_summarize("all_assets", all_assets, asset_vid)

Page 1: fetched 1000 rows (running total=1000)
Page 2: fetched 203 rows (running total=1203)
all_assets (count=1203):
  cce41cc2-8716-45e2-ae75-102289a6854b  name='Top Drive System'  description=None
  37b24df6-c4c4-4ffc-90d3-349ba346966c  name='Drilling Derrick Assembly'  description=None
  1d338ea3-7841-43fc-afe6-702966cbbffc  name='Seawater Lift Pump B'  description='Centrifugal seawater lift pump train B'
  4d839ff3-3a8f-4258-81d5-08409cdf2475  name='Portable Gas Detector'  description='Portable multi-gas detector for confined space'
  a86482db-1947-43e7-9781-67c6bc38ad09  name='MV Motor 1st Stage Compressor Drive'  description='Medium voltage motor for 1st stage compressor'
  e3e89763-dd36-4823-9738-17d9a4fecd02  name='PA/GA System Process Area'  description='Public address and general alarm system'
  056df394-acc1-4b71-9897-0053e2aec3b0  name='Hydraulic Torque Wrench 3/4in'  description='Hydraulic torque wrench for flange bolting'
  2a804545-b201-4985-822f-45c14ac0d1a5  name='Ele

### Name-only full-text search

This cell runs `instances.search` with scoring restricted to the `name` property (`properties=["name"]`). It demonstrates a tighter and usually faster search profile than scoring across all indexed text fields.

In [7]:
# Search by partial name, scoring only on `name` for tighter relevance.
sample = client.data_modeling.instances.list(sources=asset_vid, space=INST_SP, limit=10)
sample_name = next(
    ((n.properties.get(asset_vid, {}) or {}).get("name") for n in sample if (n.properties.get(asset_vid, {}) or {}).get("name")),
    None,
)
if not sample_name:
    raise RuntimeError("No assets with name found in INST_SP")

search_term = sample_name[:-2] if len(sample_name) > 2 else sample_name
hits = client.data_modeling.instances.search(
    view=asset_vid,
    space=INST_SP,
    query=search_term,
    properties=["name"],
    limit=25,
)

print(f"Name-only search for {search_term!r}")
_summarize("hits", hits, asset_vid)

Name-only search for 'Electrical Distributi'
hits (count=1):
  VAL-PH-23-EL  name='Electrical Distribution'  description='Power generation and distribution'


### Search + filter using real source context

This cell combines full-text search (`query`) with a hard filter (`sourceContext`) discovered from real data. It demonstrates how to constrain relevance search to a known slice of the dataset for better precision and performance.

In [8]:
# Search + filter: partial name scoped to a real sourceContext value.
from collections import Counter

probe = client.data_modeling.instances.list(sources=asset_vid, space=INST_SP, limit=200)
source_context = next(
    (val for val, _ in Counter(((n.properties.get(asset_vid, {}) or {}).get("sourceContext") for n in probe)).most_common() if val),
    None,
)
if source_context is None:
    raise RuntimeError("No populated sourceContext found in INST_SP")

token = (sample_name[:4] if sample_name else "")
scoped_hits = client.data_modeling.instances.search(
    view=asset_vid,
    space=INST_SP,
    query=token,
    properties=["name"],
    filter=Equals(asset_vid.as_property_ref("sourceContext"), source_context),
    limit=25,
)

print(f"Scoped search (query={token!r}, sourceContext={source_context!r})")
_summarize("scoped_hits", scoped_hits, asset_vid)

Scoped search (query='Elec', sourceContext='cfihos_test')
scoped_hits (count=3):
  2a804545-b201-4985-822f-45c14ac0d1a5  name='Electrical Equipment Room'  description='Main electrical equipment room enclosure'
  VAL-PH-23-EL  name='Electrical Distribution'  description='Power generation and distribution'
  23-EN-8001  name='Electrical Equipment Room'  description='Main electrical equipment room enclosure'


### Archived duplicate (merged into constrained search)

This section is intentionally archived to keep the notebook lean. Use the canonical **Search + filter using real source context** pattern above.

In [9]:
# Archived duplicate.
# This constrained-search variant was merged into the canonical
# "Search + filter using real source context" example above.

### Top-K search -> related retrieval (two-phase hydrate)

Use search to rank and cap the entry set, then retrieve related entities from those anchors.
This avoids expensive broad traversals when users only need the most relevant matches.

In [10]:
# Phase 1: search top-K tags by name token.
seed = client.data_modeling.instances.list(sources=asset_vid, space=INST_SP, limit=10)
seed_name = next(((n.properties.get(asset_vid, {}) or {}).get("name") for n in seed if (n.properties.get(asset_vid, {}) or {}).get("name")), None)
if not seed_name:
    raise RuntimeError("No Tag names available for top-K search demo")

query_token = seed_name[:4] if len(seed_name) >= 4 else seed_name
top_hits = client.data_modeling.instances.search(
    view=asset_vid,
    space=INST_SP,
    query=query_token,
    properties=["name"],
    limit=10,
)
print(f"Top-K tag hits for {query_token!r}")
_summarize("top_hits", top_hits, asset_vid)

# Phase 2: use hit references to retrieve related time series.
asset_refs = [DirectRelationReference(n.space, n.external_id) for n in top_hits]
if not asset_refs:
    print("No top hits found, skipping related retrieval")
else:
    ts_related = client.data_modeling.instances.search(
        view=ts_vid,
        space=INST_SP,
        query=None,
        filter=ContainsAny(ts_vid.as_property_ref("assets"), asset_refs),
        limit=100,
    )
    _summarize("related_timeseries", ts_related, ts_vid, extra_fields=["unit"])

Top-K tag hits for 'Elec'
top_hits (count=3):
  2a804545-b201-4985-822f-45c14ac0d1a5  name='Electrical Equipment Room'  description='Main electrical equipment room enclosure'
  VAL-PH-23-EL  name='Electrical Distribution'  description='Power generation and distribution'
  23-EN-8001  name='Electrical Equipment Room'  description='Main electrical equipment room enclosure'
related_timeseries (count=0):


### Query relaxation fallback (strict -> broad)

Production search often needs a fallback when strict filters return zero hits.
Try strict constraints first, then relax while keeping output bounded and explicit.

In [11]:
# Strict search: query + sourceContext. If 0 hits, retry broader query-only search.
probe = client.data_modeling.instances.list(sources=asset_vid, space=INST_SP, limit=100)
strict_ctx = next((((n.properties.get(asset_vid, {}) or {}).get("sourceContext")) for n in probe if ((n.properties.get(asset_vid, {}) or {}).get("sourceContext"))), None)
strict_name = next((((n.properties.get(asset_vid, {}) or {}).get("name")) for n in probe if ((n.properties.get(asset_vid, {}) or {}).get("name"))), None)
if not strict_name:
    raise RuntimeError("No names available for relaxation demo")
strict_token = strict_name[:4] if len(strict_name) >= 4 else strict_name

strict_hits = client.data_modeling.instances.search(
    view=asset_vid,
    space=INST_SP,
    query=strict_token,
    properties=["name"],
    filter=Equals(asset_vid.as_property_ref("sourceContext"), strict_ctx) if strict_ctx else None,
    limit=25,
)

if len(strict_hits) > 0:
    print("Strict search succeeded")
    _summarize("strict_hits", strict_hits, asset_vid)
else:
    print("Strict search returned 0 hits; retrying broader query-only search")
    broad_hits = client.data_modeling.instances.search(
        view=asset_vid,
        space=INST_SP,
        query=strict_token,
        properties=["name"],
        limit=25,
    )
    _summarize("broad_hits", broad_hits, asset_vid)

Strict search succeeded
strict_hits (count=3):
  2a804545-b201-4985-822f-45c14ac0d1a5  name='Electrical Equipment Room'  description='Main electrical equipment room enclosure'
  VAL-PH-23-EL  name='Electrical Distribution'  description='Power generation and distribution'
  23-EN-8001  name='Electrical Equipment Room'  description='Main electrical equipment room enclosure'


### Archived duplicate (merged into search basics)

This variant is merged into the canonical **Name-only full-text search** example to reduce overlap.

In [12]:
# Archived duplicate.
# This "discover name then search" pattern is covered by the
# canonical "Name-only full-text search" cell above.

### Archived duplicate (merged into top-K hydrate)

This two-step related-retrieval pattern is covered by the canonical **Top-K search -> related retrieval (two-phase hydrate)** section.

In [13]:
# Archived duplicate.
# This two-phase search->related retrieval pattern is now represented by the
# canonical "Top-K search -> related retrieval (two-phase hydrate)" example.

## Archived iterative listing (merged into cursor pagination)

Using instances API you can fetch the instances in batches, to avoid timeouts and reduce memory load

In [14]:
# Archived duplicate.
# For full retrieval at scale, use the canonical cursor-paginated /sync example:
# "List all assets safely (cursor pagination, no functional filter)".

# Assets
## General asset queries

### Archived list-based variant (lean flow uses search-first + cursor pagination)

Listing assets works almost the same as in the case of legacy assets. The main difference is the **sources** argument, that allows to choose the properties that will be fetched, by selecting a view (or a list of views).

You can sort/filter either by using a property specified within a View or node/edge registry.
Sorting by created/updated time is not allowed as of now, due to performance considerations (too much reindexing on every instance update).

In [15]:
# Archived duplicate.
# This broad list-based discovery example is superseded by the lean search-first
# section and the cursor-paginated all-assets retrieval example.

### Archived query-asset listing variant (merged into lean search/constrained flow)

Use `/query` when you need only a specific subset of properties of the retrieved instances. This query is equivalent to the `list()` example above.

In [16]:
# Archived duplicate.
# This query-based asset listing variant is merged into the lean search +
# constrained-search examples and kept out to reduce overlap.


# Get subtree of an asset

Getting a subset of assets based on a root is a common use case. Use the 'path' property to extract all assets with a given node in their paths.

### Archived contrast: ContainsAll (prefer Prefix below)

In [17]:
# Archived contrast example.
# ContainsAll-on-path is kept out of the lean flow; prefer the canonical
# Prefix-based subtree retrieval example below for production patterns.

### With Prefix (recommended)

`Prefix` on `path` uses the prefix index and is typically much faster than `ContainsAll` for subtree extraction. Prefer `Prefix` whenever you have the root's full `path`. Use `ContainsAll` only when you need to match a node at an arbitrary position in the path.

In [18]:
# Reuse sub_tree_root if the ContainsAll cell above has been run; otherwise discover one.
try:
    sub_tree_root
except NameError:
    probe = client.data_modeling.instances.list(sources=asset_vid, space=INST_SP, limit=200)
    sub_tree_root = None
    for node in probe:
        parent = (node.properties.get(asset_vid, {}) or {}).get("parent")
        if parent and parent.get("space") and parent.get("externalId"):
            sub_tree_root = NodeId(parent["space"], parent["externalId"])
            break
    if sub_tree_root is None:
        raise RuntimeError(
            "No Tag in INST_SP has a populated `parent` - cannot derive a subtree root."
        )
    print(f"Using subtree root: {sub_tree_root}")

# Retrieve the root asset first to get its path (excluded from the timing below).
sub_tree_root_retrieved = client.data_modeling.instances.retrieve_nodes(
    sub_tree_root,
    sources=asset_vid,
)

start_time = time.time()
sub_tree_nodes_prefix = client.data_modeling.instances.list(
    sources=asset_vid,
    filter=Prefix(
        property=asset_vid.as_property_ref("path"),
        value=sub_tree_root_retrieved.properties.data[asset_vid]["path"],
    ),
    limit=500,
)
prefix_time = time.time() - start_time
print(f"Prefix filter call took: {prefix_time:.3f} seconds")
_summarize("sub_tree_nodes_prefix", sub_tree_nodes_prefix, asset_vid)

Using subtree root: NodeId(space='inst_location', external_id='VAL-PH-23')
Prefix filter call took: 0.132 seconds
sub_tree_nodes_prefix (count=90):
  VAL-PH-23-EL  name='Electrical Distribution'  description='Power generation and distribution'
  VAL-PH-23-GC  name='Gas Compression System'  description='1st 2nd 3rd stage gas compression'
  VAL-PH-23-UT  name='Utilities System'  description='General utilities and support'
  VAL-PH-23-WT  name='Water Treatment System'  description='Seawater and chemical injection'
  626221d1-e8f3-4761-97bd-239e013c7f75  name='Module Support Structure'  description='Compressor module support steel'
  6b0eb178-2c4f-4fca-901b-2c59bf812582  name='Instrument Equipment Room'  description='Instrument equipment room enclosure'
  a4b91d72-0d1b-4902-a8d4-98dfcbeabf70  name='1st Stage Safety Relief Valve'  description='Pressure safety valve 1st stage discharge'
  cd5759fc-0fa0-4ce3-9831-47529722f738  name='2nd Stage Suction ESDV'  description='Emergency shutdown val

## Archived duplicate: single-asset representation examples

As you know, a single instance may have its properties in multiple views. When querying, listing or retrieval, it's possible to get multiple sources (views) along with their properties.

In [19]:
# Archived duplicate.
# Multi-view single-node retrieval is omitted in the lean flow.
# Keep the graph traversal and search-first examples as primary patterns.

### Archived duplicate SDK variant

In [20]:
# Archived duplicate.
# Query-SDK single-asset lookup removed from lean flow to reduce overlap
# with search-first and graph traversal sections.


## Get parent and/or children of an asset

### Note that you can use this logic for any kind of **single** direct relations (and their reverse). For example, you can retrieve the type of the asset (see below)

This way, you can traverse a graph using direct relations

In [21]:
# asset_eid = "WLL-6080740225"
asset_eid = "23-XX-9105"
# Tip: parameterise space with Equals(("node","space"), {"parameter": "space"}) - SpaceFilter
# does not currently accept {"parameter": "..."}.
ASSET_PROPS = ["name", "description", "source", "parent", "tags"]
query = Query(
    with_={  # FROM all Nodes WHERE space = INST_SP and externalId = CNY-AC
        "asset": NodeResultSetExpression(
            filter=And(
                Equals(property=("node", "externalId"), value=asset_eid),
                Equals(property=("node", "space"), value={"parameter": "space"}),
            ),
        ),
        "parent": NodeResultSetExpression(
            from_="asset",
            through=asset_vid.as_property_ref("parent"),
            direction="outwards",
        ),
        "children": NodeResultSetExpression(
            from_="asset",
            through=asset_vid.as_property_ref("parent"),
            direction="inwards",
        ),
        "further_children": NodeResultSetExpression(
            from_="children",
            through=asset_vid.as_property_ref("parent"),
            direction="inwards",
        ),
        "type": NodeResultSetExpression(
            from_="asset",
            through=asset_vid.as_property_ref("type"),
            direction="outwards",
        ),
    },
    select={
        "asset": Select([SourceSelector(source=asset_vid, properties=ASSET_PROPS)]),
        "parent": Select([SourceSelector(source=asset_vid, properties=ASSET_PROPS)]),
        "children": Select([SourceSelector(source=asset_vid, properties=ASSET_PROPS)]),
        "further_children": Select([SourceSelector(source=asset_vid, properties=ASSET_PROPS)]),
        "type": Select([SourceSelector(source=asset_vid, properties=ASSET_PROPS)]),
    },
    parameters={"space": INST_SP},
)
try:
    res = run_query(client, query)
    for key in ("asset", "parent", "children", "further_children", "type"):
        _summarize(key, res[key], asset_vid)
except CogniteAPIError as e:
    log_api_error(e)



asset (count=1):
  23-XX-9105  name='23-XX-9105'  description='VRD - 1ST STAGE SUCTION/DISCHARGE COOLER SKID'
parent (count=1):
  23-1ST STAGE COMPRESSION-PH  name='23-1ST STAGE COMPRESSION-PH'  description='1ST STAGE COMPRESSION ON PH'
children (count=4):
  90-MX-9245  name='90-MX-9245'  description='VRD - MONORAIL OVER WALKWAY FOR EQUIPMENT ON SKID 23-XX-9105'
  23-HA-9103  name='23-HA-9103'  description='VRD - 1ST STAGE SUCTION COOLER'
  23-HA-9114  name='23-HA-9114'  description='VRD - 1ST STAGE DISCHARGE COOLER 1'
  23-HA-9115  name='23-HA-9115'  description='VRD - 1ST STAGE DISCHARGE COOLER 2'
further_children (count=44):
  23-PDT-92501  name='1st Stage Suction Cooler dP Transmitter'  description='Differential pressure transmitter suction cooler'
  45-HV-92609  name='45-HV-92609'  description='VRD - PH 1STSTGDISCCOOL COOLMED OUT'
  45-PSV-92613  name='45-PSV-92613'  description='VRD - PH 1STSTGDISCCOOL SHELL'
  45-TT-92607B  name='45-TT-92607B'  description='VRD - PH 1STSTGDISCCL

## Using the Nested filter

Nested filter allows to use property of the directly related View to filter the instances. The filter can be applied only to single direct relations. 

In [22]:
query = Query(
    with_={
        "asset": NodeResultSetExpression(
            # Tags whose PARENT TAG has 'functional-location' in its tags property.
            # Why parent? Nested can only filter through single-valued direct relations.
            # On Tag, `parent` is the only single DR to another Tag.
            filter=Nested(
                scope=asset_vid.as_property_ref("parent"),
                filter=ContainsAll(
                    property=asset_vid.as_property_ref("tags"),
                    values=["functional-location"],
                ),
            ),
            limit=500,
        ),
        "equipment": NodeResultSetExpression(
            # FROM all Equipment WHERE asset (a direct relation to a Tag) has sourceContext == "cfihos_test"
            filter=Nested(
                scope=eq_vid.as_property_ref("asset"),
                filter=Equals(
                    property=asset_vid.as_property_ref("sourceContext"),
                    value="cfihos_test",
                ),
            ),
            limit=500,
        ),
    },
    select={
        "asset": Select(
            [SourceSelector(source=asset_vid, properties=["name", "description", "tags", "parent"])]
        ),
        "equipment": Select(
            [SourceSelector(source=eq_vid, properties=["name", "description", "tags", "asset"])]
        ),
    },
)
try:
    res = run_query(client, query)
    _summarize("asset", res["asset"], asset_vid)
    _summarize("equipment", res["equipment"], eq_vid)
except CogniteAPIError as e:
    log_api_error(e)


asset (count=98):
  VAL-PH-23-EL  name='Electrical Distribution'  description='Power generation and distribution'
  VAL-PH-23-GC  name='Gas Compression System'  description='1st 2nd 3rd stage gas compression'
  VAL-PH-23-UT  name='Utilities System'  description='General utilities and support'
  VAL-PH-23-WT  name='Water Treatment System'  description='Seawater and chemical injection'
  VAL-PH-48  name='Area 48 Subsea Systems'  description='Subsea systems area 48'
  626221d1-e8f3-4761-97bd-239e013c7f75  name='Module Support Structure'  description='Compressor module support steel'
  6b0eb178-2c4f-4fca-901b-2c59bf812582  name='Instrument Equipment Room'  description='Instrument equipment room enclosure'
  a4b91d72-0d1b-4902-a8d4-98dfcbeabf70  name='1st Stage Safety Relief Valve'  description='Pressure safety valve 1st stage discharge'
  cd5759fc-0fa0-4ce3-9831-47529722f738  name='2nd Stage Suction ESDV'  description='Emergency shutdown valve 2nd stage suction'
  0192bf8a-c9a6-4f47-b491-2

### Separator: SDK query vs raw JSON query

This short cell marks the transition between the SDK-based `Query(...)` example and the equivalent raw JSON `/query` payload example below. Use it as a visual separator when reading the notebook flow.

In [23]:
# Archived separator cell (no runtime logic).

## Archived duplicate: raw JSON payload variant

In some cases you may need to use a json object instead of SDK for querying

In [24]:
# Archived duplicate.
# Raw JSON payload variant removed from lean flow.
# Use the SDK Query example in the graph traversal section instead.


# Archived duplicate: additional related-entity examples
## Retrive timeseries related to an asset
Activities and files can be returned the same way.

The main problem here is that there is no way to extract assets and then use them to find the related timeseries. It is not possible because
- the properties holding node references pointing to assets are lists of direct relations
- reverse lists of direct relations cannot be queried

If your use case requires traversing multiple nodes both ways and lists of direct relations do not fulfill the requirements - that's when you need edges. Another way is to chain the queries outside of 'query' structure (query -> get result -> use in next query)

In [25]:
# Archived duplicate.
# Asset -> related timeseries retrieval is already covered by the
# top-K hydrate search example and graph traversal examples.


## Retrieve activities of a timeseries and equipment related to these activities

In [26]:
# Replace external_id with a real TimeSeriesData externalId from your project.
timeseries_id = NodeId(space=INST_SP, external_id="TS-XV92501-POS")
query = Query(
    with_={
        "activities": NodeResultSetExpression(
            filter=ContainsAll(
                property=wo_vid.as_property_ref("timeSeries"),
                values={"parameter": "timeseries"},
            ),
            limit=100,
        ),
        "equipment_activities": NodeResultSetExpression(
            from_="activities",
            through=wo_vid.as_property_ref("equipment"),  # must be a property reference
            limit=10,
        ),
    },
    select={
        "activities": Select(
            [
                SourceSelector(
                    source=wo_vid,
                    properties=["name", "description", "source", "assets", "equipment"],
                ),
            ],
        ),
        "equipment_activities": Select(
            [
                SourceSelector(
                    source=eq_vid,
                    properties=["name", "description", "source"],
                ),
            ],
        ),
    },
    parameters={"timeseries": [timeseries_id.dump(include_instance_type=False)]},
)
try:
    res = run_query(client, query)
    _summarize("activities", res["activities"], wo_vid)
    _summarize("equipment_activities", res["equipment_activities"], eq_vid)
except CogniteAPIError as e:
    log_api_error(e)

activities (count=2):
  WO-001003  name='ESDV Functional Test'  description='Periodic functional test 1st stage suction ESDV'
  OP-001003-010  name='ESDV Stroke Test'  description='Full stroke test of ESDV'
equipment_activities (count=1):
  EQ-VA-9301  name='1st Stage Suction ESDV'  description='16in emergency shutdown valve'


## Archived duplicate equipment traversal

You can retrieve equipment related to an asset through the 'asset' property in the Equipment.
This is useful when trying to get the equipment instances associated with assets of a certain type or class
or extensions of CogniteAsset with some properties.

Not that it only works with Equipment - all other Asset entity relationships (to files, timeseries, activities)
are Reverse **Lists** of direct relations, meaning they cannot be traversed inwards. 

In [27]:
# Archived duplicate.
# Equipment-by-asset-tag traversal is removed from the lean default flow.
# Keep parent/children and nested traversal examples as canonical graph patterns.


# Using the cursor

Use cursor pagination whenever a result set can exceed one page (for `/query` and `/sync`-style traversal).

Key patterns:

- Keep page limits modest and iterate until drained, instead of assuming one call is complete.
- Carry `query.cursors` forward between calls to resume safely.
- Stop when all selected result sets are empty for a page.
- For very large traversals, stream/process per page instead of accumulating everything in memory.

The next cells provide a concrete `get_data(...)` helper and an executable example.

In [28]:
# Cursor-based pagination for data model queries (inspired by the Yggdrasil team).
# Uses run_sync (retry-wrapped /sync) so transient 408/429/5xx are handled transparently.

def get_data(
    client: CogniteClient,
    query: Query,
    max_iterations: int | None = 100,
) -> tuple[dict[str, list[NodeListWithCursor | EdgeListWithCursor]], dict[str, str]]:
    """Cursor based pagination for data model queries.

    The query object's cursors are updated in-place so the same query can be resumed.

    Args:
        client: The Cognite client to use for making the query.
        query: The query to fetch data from CDF data model.
        max_iterations: Maximum number of pages to fetch. Use None or -1 for no limit.

    Returns:
        A tuple of (collected_data, final_cursors). final_cursors is empty when the
        result set is fully drained.
    """
    if any(c for c in (query.cursors or {}).values()):
        print("Cursors already set in query, continuing retrieval.")

    collected_data: dict[str, list] = defaultdict(list)
    current_iteration = 0
    if max_iterations is None or max_iterations == -1:
        max_iterations = float("inf")

    res = None
    while current_iteration < max_iterations:
        res = run_sync(client, query)

        if res is None:
            if not collected_data:
                print("No data returned, exiting loop.")
                return {}, {}
            print("Query failed, but returning collected data so far.")
            return collected_data, {}

        # Empty page across all selections = fully drained (cursor still kept for resume).
        if all(not res.data[selection] for selection in res.data):
            print("No more data available, exiting loop.")
            return collected_data, {}

        for selection in res.data:
            collected_data[selection].extend(res.data[selection])

        query.cursors = res.cursors
        current_iteration += 1

    print(f"Collected data for {current_iteration} iterations.")
    return collected_data, (res.cursors if res is not None else {})

### Example: paginating a large result set with `get_data`

The query below intentionally has a small per-page `limit` (50) so we can see
`get_data` iterate. It returns every Tag in `INST_SP` that has data in the
Asset view, walking pages until the result set is drained or `max_iterations`
is hit.

In [29]:
# Build a small-page query to demonstrate cursor pagination via get_data.
paged_query = Query(
    with_={
        "assets": NodeResultSetExpression(
            filter=And(
                Equals(["node", "space"], value=INST_SP),
                HasData(views=[asset_vid]),
            ),
            limit=50,  # small per-page limit so we actually see multiple iterations
        ),
    },
    select={
        "assets": Select(
            [SourceSelector(asset_vid, ["name", "description", "tags"])]
        ),
    },
)

try:
    collected, final_cursors = get_data(client, paged_query, max_iterations=5)
    assets = collected.get("assets", [])
    print()
    _summarize("assets (total collected)", assets, asset_vid)

    if final_cursors:
        print(f"\nMore data available - resume with paged_query.cursors = {final_cursors!r}")
    else:
        print("\nResult set fully drained.")
except CogniteAPIError as e:
    log_api_error(e)

Collected data for 5 iterations.

assets (total collected) (count=250):
  cce41cc2-8716-45e2-ae75-102289a6854b  name='Top Drive System'  description=None
  37b24df6-c4c4-4ffc-90d3-349ba346966c  name='Drilling Derrick Assembly'  description=None
  1d338ea3-7841-43fc-afe6-702966cbbffc  name='Seawater Lift Pump B'  description='Centrifugal seawater lift pump train B'
  4d839ff3-3a8f-4258-81d5-08409cdf2475  name='Portable Gas Detector'  description='Portable multi-gas detector for confined space'
  a86482db-1947-43e7-9781-67c6bc38ad09  name='MV Motor 1st Stage Compressor Drive'  description='Medium voltage motor for 1st stage compressor'
  e3e89763-dd36-4823-9738-17d9a4fecd02  name='PA/GA System Process Area'  description='Public address and general alarm system'
  056df394-acc1-4b71-9897-0053e2aec3b0  name='Hydraulic Torque Wrench 3/4in'  description='Hydraulic torque wrench for flange bolting'
  2a804545-b201-4985-822f-45c14ac0d1a5  name='Electrical Equipment Room'  description='Main ele